# T45C / E18 — Chấm theo từng câu phản hồi, gộp max thay vì trung bình

Chương 7 của quyển giải thích vì sao *chú ý nhìn đúng chỗ* (E06, 87,8 %) mà *phân loại không tốt
hơn* bằng ba lý do. Lý do thứ hai — **gộp trung bình cả phản hồi làm loãng một mệnh đề bịa** — là
lý do duy nhất kiểm được bằng thí nghiệm. E18 kiểm nó.

## Làm gì

Trích lại ViHallu đúng như E03 (cùng mô hình, cùng prompt, cùng cách chia đoạn), chỉ **thêm**
vào mỗi bản ghi các véc-tơ tính riêng trên từng câu của phản hồi. Sau đó so bốn nhánh trên
**cùng một shard**:

| Nhánh | Gộp trục phản hồi |
|---|---|
| `whole` | trung bình cả phản hồi, tức E03 |
| `mean` | trung bình trong từng câu, rồi trung bình giữa các câu |
| `max` | … rồi lấy lớn nhất giữa các câu — **nhánh có dự đoán** |
| `min` | … rồi lấy nhỏ nhất — thêm trước khi chạy, vì câu bịa làm tỷ lệ lookback *giảm* |

Mỗi nhánh tự chọn cách gộp đầu chú ý trên dev, rồi chấm test đúng một lần, giống hệt E03.

**Dự đoán ghi trước khi chạy** (khóa trong `scripts/run_sentence_pooling.py`, có ca kiểm thử):
gộp max tăng F1 lớp extrinsic và giảm số ảo giác bị đoán thành `no` so với `whole`; macro-F1 có
thể vẫn nằm trong khoảng tin cậy.

## Trần của thí nghiệm — đọc trước khi xem kết quả

**71,6 % phản hồi ViHallu chỉ có một câu** (đo 22/10, cùng hàm tách câu với ngữ cảnh). Ở đó cả
bốn nhánh cho cùng một véc-tơ, nên mọi khác biệt chỉ có thể đến từ 28 % còn lại. Script in thêm
điểm trên riêng phần phản hồi từ hai câu trở lên.

## Thời gian

| Việc | Ước tính |
|---|---|
| Trích train 5.600 mẫu | ~45 phút |
| Trích dev, test 700 mẫu mỗi tập | ~6 phút mỗi tập |
| So bốn nhánh (CPU) | ~15 phút |

**Tổng khoảng 75 phút.** Mỗi ô trích chạy lại được; mẫu nào xong là ghi xuống ngay.

## Cần tải về sau khi chạy

Ba shard `vihallu_*_8d0bb26a198b.jsonl` (khoảng 0,9 GB) và notebook đã chạy. Trên máy, đặt shard
vào `data/processed/` rồi chạy lại `python scripts/run_sentence_pooling.py`: lúc đó có shard E03
cũ bên cạnh nên script kiểm được **các trường cũ trùng từng giá trị** và ghi kết quả chính thức vào
`results/runs.jsonl`.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock (requirements-kaggle.txt, từ 22/10/2026).
# `--no-deps` cố ý không kéo phụ thuộc nào về, nên rank-bm25 (BM25 dựng ngữ cảnh) nằm trong file ghim.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây. Chặn những gì phiên này không tự tạo được.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

problems = []
packages = ("torch", "transformers", "bitsandbytes", "pandas", "sklearn")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
print(f"  gói phải có sẵn   : {'THIẾU ' + str(absent) if absent else 'đủ'}")
if absent:
    problems.append(f"thiếu gói {absent}")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ"
print(f"  GPU               : {gpu}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
try:
    raw = Path(find_raw_dir())
    present = (raw / "vihallu_train.csv").exists()
    print(f"  dữ liệu thô       : {raw} — vihallu_train.csv {'có' if present else 'KHÔNG CÓ'}")
    if not present:
        problems.append("không thấy vihallu_train.csv")
except Exception as error:
    print(f"  dữ liệu thô       : KHÔNG TÌM THẤY ({error})")
    problems.append("chưa gắn dataset dữ liệu thô")
cfg = load_config("configs/e18_sentence_pooling_vihallu.yaml")
print(f"  hash trích        : {extraction_hash(cfg)} (E03 cũ là 8c49fc0417f1, phải khác)")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

In [ ]:
# Ô 4 — chuẩn bị dữ liệu và kiểm thử trước khi tốn GPU. Khoảng 1 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_sentence_pooling.py tests/test_attention_math.py tests/test_lookback.py tests/test_chunk_features.py tests/test_assemble.py -q

## Trích đặc trưng

Dòng `thêm theo từng câu` phải hiện `có`. Cột `lỗi` phải là 0.

In [ ]:
# Ô 5 — trích tập train. Khoảng 45 phút.
!python scripts/extract_features.py --config configs/e18_sentence_pooling_vihallu.yaml --split train

In [ ]:
# Ô 6 — trích tập dev. Khoảng 6 phút. Tập này để chọn cách gộp đầu, không để báo kết quả.
!python scripts/extract_features.py --config configs/e18_sentence_pooling_vihallu.yaml --split dev

In [ ]:
# Ô 7 — trích tập test. Khoảng 6 phút.
!python scripts/extract_features.py --config configs/e18_sentence_pooling_vihallu.yaml --split test

In [ ]:
# Ô 8 — XEM TRƯỚC bốn nhánh. Khoảng 15 phút, CPU. Ghi ra file riêng, không đụng runs.jsonl.
# Số chính thức chấm trên máy (quy tắc T23: mọi phép so sánh chấm trên cùng một máy, điểm dev
# lệch tới 0,0075 giữa Kaggle và máy cá nhân), nơi có cả shard E03 cũ để kiểm tái lập.
!python scripts/run_sentence_pooling.py --results-path /kaggle/working/e18_runs_kaggle.jsonl

In [ ]:
# Ô 9 — chép shard ra /kaggle/working để tải về.
import shutil
from pathlib import Path

for path in sorted(Path("data/processed").glob("vihallu_*_8d0bb26a198b.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:.1f} MB")